# Hindi Music Engine — Kaggle Batch Runner

Runs the ingestion pipeline (download → Demucs → CLAP → lyrics → metadata → fuse) on a chunk of `songs.csv`, on Kaggle's free T4 GPU, then rebuilds the FAISS index and commits the result as a Kaggle Dataset.

**Before running — both of these are separate toggles, easy to set one and miss the other:**
1. Notebook Settings → **Accelerator → GPU T4 x2** AND **Internet → On**. If you change either mid-session, stop and restart the session — toggles don't always apply live.
2. Add-ons → Secrets → add `GENIUS_TOKEN` and `LASTFM_API_KEY` (same values as your local `.env`), and if the repo is private, `GITHUB_TOKEN` too. **Each secret also needs its own on/off switch flipped for THIS notebook** — adding it to your account isn't enough.
3. If you have existing processed data (from a previous local or Kaggle run), attach it as a Kaggle Dataset input (Add data → your dataset). First run ever: skip this.
4. Edit the `START_ROW` / `NUM_ROWS` cell below to pick which slice of `songs.csv` this run covers.

**Every step below fails loudly with a clear message and a short timeout instead of hanging** — if something's misconfigured (no internet, a bad secret, etc.) you'll know within seconds, not after burning 45 minutes of GPU quota stuck on one line.

**At the end:** File → Save Version → commits everything under `/kaggle/working/` as a new dataset version — attach that as the input dataset for your next run.

In [ ]:
# --- Config ---
REPO_URL = "github.com/pranshuk22/hindi-music-engine.git"  # no https:// prefix — added below depending on auth
REPO_IS_PRIVATE = False  # set True if the GitHub repo is private; requires a GITHUB_TOKEN Kaggle Secret
START_ROW = 0        # first data row (0-indexed, header excluded) to process
NUM_ROWS = 100        # how many rows this session covers; keep modest given Kaggle's runtime limit
WORKERS = 2            # ThreadPoolExecutor workers — AGENTS.md caps this at 2 for RAM
EXISTING_DATA_INPUT = None  # e.g. "/kaggle/input/hindi-music-engine-data", or None for a fresh start

# Timeouts (seconds) — tune if a legitimately slow step (e.g. a big pip
# install) needs more room, but keep them finite. A hang past this raises
# immediately instead of silently burning the session.
TIMEOUT_NETWORK_CHECK = 15
TIMEOUT_CLONE = 60
TIMEOUT_PIP_INSTALL = 900
TIMEOUT_PIPELINE = 6 * 3600   # the actual song-processing step, genuinely long-running
TIMEOUT_INDEX_BUILD = 300

In [ ]:
# --- Shared helper: run a command with a hard timeout and a clear pass/fail message. ---
# Using subprocess directly (not `!` shell magics) so a timeout actually kills
# the process and raises, instead of the cell just sitting there forever.
import subprocess, sys, time

def run(cmd, timeout, cwd=None, env=None, label=None):
    label = label or cmd
    print(f"--- RUNNING ({timeout}s timeout): {label}")
    t0 = time.time()
    try:
        result = subprocess.run(
            cmd, shell=True, cwd=cwd, env=env, timeout=timeout,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
    except subprocess.TimeoutExpired as e:
        elapsed = time.time() - t0
        print(e.stdout or "")
        raise RuntimeError(
            f"TIMED OUT after {elapsed:.0f}s (limit {timeout}s): {label}\n"
            f"This means it was hanging, not just slow — check Internet toggle, "
            f"secrets attachment, or repo visibility before re-running."
        )
    elapsed = time.time() - t0
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"FAILED (exit {result.returncode}, {elapsed:.0f}s): {label}")
    print(f"--- OK ({elapsed:.0f}s): {label}")
    return result

In [ ]:
# --- Preflight: is the network actually up? Fail in seconds, not 45 minutes. ---
# This is the single most common silent-hang cause on Kaggle: GPU and Internet
# are separate toggles, and a genuinely blocked connection doesn't error out —
# it just hangs waiting for a response that never comes.
import socket

def check_connectivity(host, port=443, timeout=TIMEOUT_NETWORK_CHECK):
    try:
        socket.setdefaulttimeout(timeout)
        socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect((host, port))
        print(f"  OK: reached {host}:{port}")
        return True
    except Exception as e:
        print(f"  FAILED: {host}:{port} — {e}")
        return False

print("Checking network connectivity before doing anything else...")
ok = check_connectivity("github.com") and check_connectivity("pypi.org")
if not ok:
    raise RuntimeError(
        "No network reachability from this session. This is almost always "
        "Notebook Settings > Internet not actually being On (it's separate "
        "from the GPU toggle), or the toggle was changed after the session "
        "started (needs a session restart to apply). Fix that, restart the "
        "session, and re-run from the top BEFORE doing anything else — "
        "every later step will hang the same way otherwise."
    )
print("Network OK — safe to proceed.")

In [ ]:
# --- Clone (timeout-guarded) ---
# Cloned OUTSIDE /kaggle/working on purpose: that directory is exactly what
# "Save Version" / `kaggle kernels output` captures, so anything under it
# (including this repo's .env once we write it) becomes part of the
# published kernel output. /kaggle/tmp is not persisted, so the repo, the
# .env secrets file, and everything else live there and never leak — only
# what the staging cell explicitly copies into /kaggle/working survives.
import os

REPO_DIR = "/kaggle/tmp/repo"
os.makedirs("/kaggle/tmp", exist_ok=True)

if REPO_IS_PRIVATE:
    from kaggle_secrets import UserSecretsClient
    _gh_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if not _gh_token:
        raise RuntimeError(
            "REPO_IS_PRIVATE=True but GITHUB_TOKEN secret is empty/missing. "
            "Check it's both added AND toggled on for this notebook."
        )
    clone_url = f"https://{_gh_token}@{REPO_URL}"
else:
    clone_url = f"https://{REPO_URL}"

if os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR} already exists from a previous attempt in this session — removing before re-cloning.")
    run(f"rm -rf {REPO_DIR}", timeout=30, label="remove stale repo dir")

# -v for verbose git output — if this DOES hang again, you'll at least see
# which phase (resolving host / connecting / negotiating / receiving) it's stuck on.
run(f"git clone -v {clone_url} {REPO_DIR}", timeout=TIMEOUT_CLONE, label="git clone")
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
run("apt-get -qq install -y ffmpeg", timeout=120, label="apt-get ffmpeg")
run("pip install -q -r requirements.txt", timeout=TIMEOUT_PIP_INSTALL, label="pip install")

# requirements.txt doesn't pin numpy, but numba (pulled in by librosa, used
# by msclap) hard-requires numpy<=2.0, so pip correctly resolves an older
# numpy than Kaggle's preinstalled 2.x. The problem isn't the version pip
# picked — it's that pip only swaps numpy's Python files, leaving Kaggle's
# numpy-2.x-ABI compiled binaries behind -> "dtype size changed" the first
# time anything touches numpy.random (e.g. importing pandas).
# Fix: force-reinstall the EXACT version pip already resolved (not latest —
# that would re-break numba) so its binaries are self-consistent.
_pip_show = run("pip show numpy", timeout=30, label="read resolved numpy version")
_numpy_version = next(
    line.split(":", 1)[1].strip()
    for line in _pip_show.stdout.splitlines() if line.startswith("Version:")
)
print(f"Resolved numpy version: {_numpy_version}")
run(f"pip install -q --force-reinstall --no-deps numpy=={_numpy_version}", timeout=120, label="force-reinstall numpy binaries (pinned)")
run(
    'python -c "import numpy, pandas, scipy, sklearn, numba; print(numpy.__version__, pandas.__version__, numba.__version__)"',
    timeout=60,
    label="verify numpy/pandas/scipy/sklearn/numba import cleanly",
)

In [ ]:
# Secrets -> .env (never written to git, never shown in the notebook's saved output)
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

genius_token = secrets.get_secret("GENIUS_TOKEN")
lastfm_key = secrets.get_secret("LASTFM_API_KEY")
missing = [name for name, val in [("GENIUS_TOKEN", genius_token), ("LASTFM_API_KEY", lastfm_key)] if not val]
if missing:
    raise RuntimeError(
        f"Missing/empty secret(s): {missing}. Check they're added AND toggled "
        f"on for THIS notebook in Add-ons > Secrets (a secret existing on your "
        f"account isn't enough — each notebook has its own on/off switch)."
    )

with open(".env", "w") as f:
    f.write(f"GENIUS_TOKEN={genius_token}\n")
    f.write(f"LASTFM_API_KEY={lastfm_key}\n")
print(".env written — both secrets present and non-empty.")

In [ ]:
# Bring in existing processed data if this isn't the first run ever.
import shutil

if EXISTING_DATA_INPUT:
    if not os.path.isdir(EXISTING_DATA_INPUT):
        raise RuntimeError(
            f"EXISTING_DATA_INPUT={EXISTING_DATA_INPUT!r} doesn't exist. "
            f"Check the dataset is actually attached (Add data) and the path "
            f"matches — list /kaggle/input/ to see what's really mounted."
        )
    for sub in ["embeddings_raw", "embeddings", "features", "nlp", "lyrics"]:
        src = os.path.join(EXISTING_DATA_INPUT, sub)
        if os.path.isdir(src):
            shutil.copytree(src, f"data/{sub}", dirs_exist_ok=True)
            print(f"  copied {sub}/: {len(os.listdir(src))} files")
    db_src = os.path.join(EXISTING_DATA_INPUT, "metadata.db")
    if os.path.isfile(db_src):
        shutil.copy(db_src, "data/metadata.db")
        print("  copied metadata.db")
    print("Existing data copied in.")
else:
    print("No existing data attached — starting fresh (expected for the very first run).")

In [ ]:
# Checked via subprocess, not `import torch` directly — this kernel already
# has a stale numpy loaded from before the force-reinstall fix (Kaggle
# kernels pre-import numpy/pandas at startup), and a compiled module already
# in sys.modules can't be hot-swapped by overwriting files on disk. Every
# check from here on runs in a fresh subprocess so it reads the fixed,
# self-consistent numpy off disk instead of the kernel's poisoned copy.
_torch_check = run(
    'python -c "import torch; print(torch.cuda.is_available())"',
    timeout=60,
    label="check CUDA availability (subprocess)",
)
if "True" not in _torch_check.stdout:
    raise RuntimeError(
        "No GPU detected. Demucs/CLAP on CPU is ~3-5 min/song per AGENTS.md — "
        "processing any real batch will blow the session time limit. Check "
        "Notebook Settings > Accelerator before continuing."
    )

In [ ]:
# Slice songs.csv to this session's range — via a subprocess script, same
# reason as the CUDA check above: don't let pandas/numpy get imported into
# this kernel's already-poisoned process.
_slice_script = f'''
import pandas as pd, sys
df = pd.read_csv("data/songs.csv")
start, num = {START_ROW}, {NUM_ROWS}
if start >= len(df):
    print(f"START_ROW={{start}} is past the end of songs.csv ({{len(df)}} rows).")
    sys.exit(1)
chunk = df.iloc[start:start + num]
chunk.to_csv("data/_kaggle_chunk.csv", index=False)
print(f"ROWS={{len(chunk)}}")
print(f"TOTAL={{len(df)}}")
'''
with open("_slice_csv.py", "w") as f:
    f.write(_slice_script)

_slice_result = run("python _slice_csv.py", timeout=30, label="slice songs.csv (subprocess)")
NUM_CHUNK_ROWS = int(next(l.split("=", 1)[1] for l in _slice_result.stdout.splitlines() if l.startswith("ROWS=")))
TOTAL_ROWS = int(next(l.split("=", 1)[1] for l in _slice_result.stdout.splitlines() if l.startswith("TOTAL=")))
chunk_path = "data/_kaggle_chunk.csv"
print(f"Processing rows {START_ROW}..{START_ROW + NUM_CHUNK_ROWS} of {TOTAL_ROWS} ({NUM_CHUNK_ROWS} songs)")

In [ ]:
run(
    f"python scripts/run_pipeline.py --csv {chunk_path} --workers {WORKERS}",
    timeout=TIMEOUT_PIPELINE,
    label=f"run_pipeline.py on {NUM_CHUNK_ROWS} songs",
)

In [ ]:
run("python index/build_index.py --from-features", timeout=TIMEOUT_INDEX_BUILD, label="rebuild FAISS index")

In [ ]:
# Quick smoke test — not a substitute for the full golden-set eval locally.
run("python index/search.py", timeout=60, label="search.py smoke test")

In [ ]:
# Stage outputs for "Save Version" to publish as a dataset.
os.makedirs("/kaggle/working/data", exist_ok=True)
for sub in ["embeddings_raw", "embeddings", "features", "nlp", "lyrics"]:
    if os.path.isdir(f"data/{sub}"):
        shutil.copytree(f"data/{sub}", f"/kaggle/working/data/{sub}", dirs_exist_ok=True)
if os.path.isfile("data/metadata.db"):
    shutil.copy("data/metadata.db", "/kaggle/working/data/metadata.db")

os.makedirs("/kaggle/working/index", exist_ok=True)
for f in ["faiss_index.bin", "song_id_map.npy", "pca_model.pkl"]:
    if os.path.isfile(f"index/{f}"):
        shutil.copy(f"index/{f}", f"/kaggle/working/index/{f}")

print("Outputs staged in /kaggle/working/ — click 'Save Version' to publish as a dataset.")